# Vertex AI Vector Search Lab
Complete, tested end-to-end pipeline for semantic book search.

In [ ]:
# Step 1: Install required Google Cloud SDK packages (run once)
%pip install -q --upgrade google-cloud-aiplatform google-cloud-storage google-cloud-bigquery

In [1]:
# Step 2: Initialize Vertex AI & Configuration
import uuid
from google.cloud import aiplatform

PROJECT_ID = "jovial-coral-423517-h8"
LOCATION = "us-central1"
BUCKET_NAME = "tuc-vector-search-bucket-1"
BLOB_NAME = "books.txt"

aiplatform.init(project=PROJECT_ID, location=LOCATION)
print("Vertex AI initialized successfully.")

Vertex AI initialized successfully.


In [2]:
# Step 3: Create GCS Bucket, Upload books.txt, and Load into Memory
import os
from google.cloud import storage

LOCAL_FILE_PATH = r"C:\Users\hmira\GitHub-Org\teachucode-courses\Google-Gemini-ADK-Agents-Lab\Lab18-vector-search\books.txt"

storage_client = storage.Client(project=PROJECT_ID)

# 1. Create bucket if it doesn't exist
bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(bucket, location=LOCATION)
    print(f"Bucket '{BUCKET_NAME}' created in {LOCATION}.")
else:
    print(f"Bucket '{BUCKET_NAME}' ready.")

# 2. Upload books.txt if not already present
blob = bucket.blob(BLOB_NAME)
if not blob.exists() and os.path.exists(LOCAL_FILE_PATH):
    blob.upload_from_filename(LOCAL_FILE_PATH)
    print(f"Uploaded '{LOCAL_FILE_PATH}' to GCS.")

# 3. Download text and clean lines
text_data = blob.download_as_text()
raw_lines = [line.strip() for line in text_data.split("\n") if line.strip()]
documents = [doc for doc in raw_lines if doc != "Book-Title"]

print(f"Loaded {len(documents)} books ready for embedding!")

Bucket 'tuc-vector-search-bucket-1' ready.
Loaded 499 books ready for embedding!


In [3]:
# Step 4: Generate Embeddings in Batches (Prevents 429 Quota Exceeded)
import time
from vertexai.language_models import TextEmbeddingModel

embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")

batch_size = 100
embeddings = []

print(f"Generating embeddings for {len(documents)} books in batches of {batch_size}...")
for i in range(0, len(documents), batch_size):
    batch = documents[i : i + batch_size]
    print(f"Embedding batch {i // batch_size + 1}: books {i + 1} to {min(i + batch_size, len(documents))}...")
    response = embedding_model.get_embeddings(batch)
    embeddings.extend([emb.values for emb in response])
    if i + batch_size < len(documents):
        time.sleep(1.0)

print(f"\nSuccess! Created {len(embeddings)} embeddings of dimension {len(embeddings[0])}.")

c:\Users\hmira\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\vertexai\_model_garden\_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Generating embeddings for 499 books in batches of 100...
Embedding batch 1: books 1 to 100...
Embedding batch 2: books 101 to 200...
Embedding batch 3: books 201 to 300...
Embedding batch 4: books 301 to 400...
Embedding batch 5: books 401 to 499...

Success! Created 499 embeddings of dimension 768.


In [4]:
# Step 5: Create Vector Search Index (Tree-AH with Leaf Node Config)
# Note: Takes ~15-25 minutes to complete
index_id = f"my-book-index-{uuid.uuid4().hex[:6]}"
print(f"Creating index: {index_id}")

index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=index_id,
    dimensions=len(embeddings[0]),          # 768
    approximate_neighbors_count=10,         # Required for Tree-AH
    distance_measure_type="COSINE_DISTANCE",
    leaf_node_embedding_count=500,          # Fixes 400 algorithmConfig error
    leaf_nodes_to_search_percent=10,        # Fixes 400 algorithmConfig error
    shard_size="SHARD_SIZE_SMALL",          # Cost optimization
    index_update_method="STREAM_UPDATE",    # Allows real-time upserts
    sync=True
)

print("Waiting for index creation to complete...")
index.wait()
print("Index created successfully!")

Creating index: my-book-index-9cedd4
Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/400638557780/locations/us-central1/indexes/6605010439631798272/operations/1843978061746274304
MatchingEngineIndex created. Resource name: projects/400638557780/locations/us-central1/indexes/6605010439631798272
To use this MatchingEngineIndex in another session:
index = aiplatform.MatchingEngineIndex('projects/400638557780/locations/us-central1/indexes/6605010439631798272')
Waiting for index creation to complete...
Index created successfully!


In [ ]:
# Step 6: Create Endpoint and Deploy Index
# Note: Takes ~15-30 minutes to complete
endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name=f"{index_id}-endpoint",
    public_endpoint_enabled=True,
)

print("Deploying index to endpoint (please wait)...\n")
endpoint.deploy_index(
    index=index,
    deployed_index_id="my_unique_deployment_id_v2"
)
print("Index successfully deployed to endpoint!")

Creating MatchingEngineIndexEndpoint
Create MatchingEngineIndexEndpoint backing LRO: projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952/operations/5911150737726898176
MatchingEngineIndexEndpoint created. Resource name: projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952
To use this MatchingEngineIndexEndpoint in another session:
index_endpoint = aiplatform.MatchingEngineIndexEndpoint('projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952')
Deploying index to endpoint (please wait)...

Deploying index MatchingEngineIndexEndpoint index_endpoint: projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952
Deploy index MatchingEngineIndexEndpoint index_endpoint backing LRO: projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952/operations/4839610685761519616
MatchingEngineIndexEndpoint index_endpoint Deployed index. Resource name: projects/400638557780/locations/us-cen

In [6]:
# Step 7: Upsert Book Datapoints into the Live Index
from google.cloud.aiplatform_v1.types import index as index_types

datapoints = [
    index_types.IndexDatapoint(
        datapoint_id=str(i),
        feature_vector=emb,
    )
    for i, emb in enumerate(embeddings)
]

print(f"Upserting {len(datapoints)} vectors into the index...")
index.upsert_datapoints(datapoints=datapoints)
print(f"Successfully upserted {len(datapoints)} vectors!")

Upserting 499 vectors into the index...
Upserting datapoints MatchingEngineIndex index: projects/400638557780/locations/us-central1/indexes/6605010439631798272
MatchingEngineIndex index Upserted datapoints. Resource name: projects/400638557780/locations/us-central1/indexes/6605010439631798272
Successfully upserted 499 vectors!


In [ ]:
# Step 8: Perform Semantic Search
query_text = "find similar to Sci-Fi"
query_emb = embedding_model.get_embeddings([query_text])[0].values

# Search the 5 nearest neighbors
neighbors_list = endpoint.find_neighbors(
    deployed_index_id="my_unique_deployment_id_v2",
    queries=[query_emb],
    num_neighbors=5,
)

print(f"Query: '{query_text}'\nTop Matches:")
for neighbor in neighbors_list[0]:
    idx = int(neighbor.id)
    similarity = (1 - neighbor.distance) * 100
    print(f"- {documents[idx]} (Distance: {neighbor.distance:.4f} | Match: {similarity:.1f}%)")

Query: 'find similar to Bible'
Top Matches:
- The Testament (Distance: 0.4719 | Match: 52.8%)
- The Gospel of Judas: A Novel (Distance: 0.4900 | Match: 51.0%)
- Poisonwood Bible Edition Uk (Distance: 0.4972 | Match: 50.3%)
- Chocolate Jesus (Distance: 0.5041 | Match: 49.6%)
- Das Buch der lÃ?Â¤cherlichen Liebe. (Distance: 0.5096 | Match: 49.0%)


In [8]:
# Step 9: (Optional) Inspect Raw Vectors Stored in the Live Endpoint
stored_points = endpoint.read_index_datapoints(
    deployed_index_id="my_unique_deployment_id_v2",
    ids=["0", "1"]
)

for pt in stored_points:
    print(f"ID: {pt.datapoint_id} | Vector Dimension: {len(pt.feature_vector)} | First 5 values: {list(pt.feature_vector)[:5]}")

ID: 0 | Vector Dimension: 768 | First 5 values: [0.00255410629324615, 0.02955915965139866, 0.03797339275479317, -0.00521567277610302, 0.052736252546310425]
ID: 1 | Vector Dimension: 768 | First 5 values: [-0.048507921397686005, -0.010563786141574383, -0.002004913752898574, -0.018677735701203346, 0.045170240104198456]


In [9]:
# Step 10: Complete Cleanup (Stops All Ongoing Charges)
# IMPORTANT: Run this when you are done to avoid ~$15-$20/day endpoint billing!
print("Undeploying and deleting endpoint...")
endpoint.delete(force=True)
print("Endpoint deleted.")

print("Deleting index...")
index.delete()
print("Index deleted. All billing stopped!")

Undeploying and deleting endpoint...
Undeploying MatchingEngineIndexEndpoint index_endpoint: projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952
Undeploy MatchingEngineIndexEndpoint index_endpoint backing LRO: projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952/operations/5863014118662864896
MatchingEngineIndexEndpoint index_endpoint undeployed. Resource name: projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952
Deleting MatchingEngineIndexEndpoint : projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952
MatchingEngineIndexEndpoint deleted. . Resource name: projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952
Deleting MatchingEngineIndexEndpoint resource: projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952
Delete MatchingEngineIndexEndpoint backing LRO: projects/400638557780/locations/us-central1/indexEndpoints/3251148131194109952/